# Agente RAG en producción: Colab + Drive + Gradio + Meta API

El mismo sistema agéntico de la [actividad base](https://github.com/JheisonMB/ml/blob/main/langgraph-rag-referencias-astro/langgraph_rag_agente_referencias.ipynb) (reescribir -> recuperar/contar -> generar con citas), en cuaderno autocontenido con interfaz Gradio en vivo y streaming. El índice ya viene construido en la carpeta compartida (lo arma el otro cuaderno); este solo lo trae al disco local, pregunta y expone la UI.

In [1]:
%pip install -q gradio langchain langgraph langchain-chroma langchain-huggingface langchain-openai langchain-text-splitters sentence-transformers pypdf openai pandas gdown

## 1. Carpeta compartida + índice + clave

Monta Drive y ubica `rag_corpus` (o descarga la copia pública). Trae el índice ya construido al disco local y define `META_API_KEY` desde Secrets.

In [2]:
import json
import os, shutil, subprocess
from pathlib import Path

DEST = '/content/rag_corpus'
FOLDER_URL = 'https://drive.google.com/drive/folders/1dc94vK-VU1SOLpiQ8aHteH7Vah21n3_s'

from google.colab import drive
drive.mount('/content/drive')

candidates = ['/content/drive/MyDrive/rag_corpus']
found = subprocess.run(['find', '/content/drive/MyDrive', '-maxdepth', '3',
                        '-type', 'd', '-name', 'rag_corpus'],
                       capture_output=True, text=True).stdout.split()
candidates += [c for c in found if c not in candidates]
BASE = next((c for c in candidates if os.path.exists(f'{c}/pdfs')), None)
if BASE is None:
    if not os.path.exists(f'{DEST}/pdfs'):
        print('Descargando carpeta compartida...')
        import gdown
        gdown.download_folder(FOLDER_URL, output=DEST, quiet=False)
    else:
        print('Carpeta ya descargada en', DEST)
    BASE = DEST
    DRIVE_INDEX = None
else:
    print('Usando carpeta en Drive:', BASE)
    DRIVE_INDEX = f'{BASE}/index'

LOCAL_INDEX = Path('/content/rag_index')
def _n_files(idx):
    m = Path(idx) / 'manifest.json'
    if not m.exists():
        return -1
    try:
        return len(json.loads(m.read_text()).get('indexed', {}))
    except Exception:
        return -1
if DRIVE_INDEX and _n_files(DRIVE_INDEX) > _n_files(LOCAL_INDEX):
    print(f'Trayendo indice al disco local (Drive: {_n_files(DRIVE_INDEX)}, local: {_n_files(LOCAL_INDEX)})...')
    shutil.rmtree(LOCAL_INDEX, ignore_errors=True)
    shutil.copytree(DRIVE_INDEX, LOCAL_INDEX)
    print('indice listo en local')

os.environ['RAG_PDFS'] = f'{BASE}/pdfs'
os.environ['RAG_INDEX'] = str(LOCAL_INDEX)

if not os.environ.get('META_API_KEY'):
    try:
        from google.colab import userdata
        os.environ['META_API_KEY'] = userdata.get('META_API_KEY')
    except Exception:
        from getpass import getpass
        os.environ['META_API_KEY'] = getpass('META_API_KEY: ')

manifest = json.loads((LOCAL_INDEX / 'manifest.json').read_text()) if (LOCAL_INDEX / 'manifest.json').exists() else {'indexed': {}, 'failed': {}}
print('PDFs:', os.environ['RAG_PDFS'])
print('Indice:', os.environ['RAG_INDEX'])
print('archivos indexados:', len(manifest['indexed']), '| fallidos:', len(manifest.get('failed', {})))


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Usando carpeta en Drive: /content/drive/MyDrive/rag_corpus
Trayendo indice al disco local...
indice listo en local
PDFs: /content/drive/MyDrive/rag_corpus/pdfs
Indice: /content/rag_index
archivos indexados: 47 | fallidos: 0


## 2. Agente con herramientas

Un agente LangChain decide solo: `buscar_documentos` y `contar_documentos`. Sin intents ni modos cableados. Prueba de conteo y de búsqueda.

In [3]:
import json
import re
from pathlib import Path
from langchain_chroma import Chroma
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage
from langchain_core.tools import tool
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langgraph.checkpoint.memory import MemorySaver

META_MODEL = os.environ.get('META_MODEL', 'muse-spark-1.3-contributor')
EMBED_MODEL = os.environ.get('RAG_EMBED_MODEL', 'sentence-transformers/all-MiniLM-L6-v2')
DEFAULTS = {'temperature': 0.0, 'variants': 3, 'k': 4,
            'sim_min': float(os.environ.get('RAG_SIM_MIN', 0.35)), 'max_tokens': 2048}

vectorstore = Chroma(collection_name='rag_core_docs_cosine',
    embedding_function=HuggingFaceEmbeddings(model_name=EMBED_MODEL),
    persist_directory=os.environ['RAG_INDEX'], collection_metadata={'hnsw:space': 'cosine'})
print('chunks en el indice:', vectorstore._collection.count())

CITATION = re.compile(r'[\(\[]([\w\-\.]+\.pdf),\s*p\.\s*(\d+)[\)\]]')
_req = {'k': 4, 'sim_min': 0.35}

def tag(doc):
    return f"({doc.metadata['source']}, p. {doc.metadata['page']})"

@tool
def buscar_documentos(consultas: list, anio: str = ''):
    '''Busca fragmentos en los documentos con varias consultas en ingles tecnico. anio (cuatro digitos) es opcional y filtra por año. Devuelve fragmentos con (archivo.pdf, p. N). Si no hay nada, devuelve texto vacio.'''
    scope = {'year': anio} if anio else None
    best = {}
    for query in consultas:
        for doc, distance in vectorstore.similarity_search_with_score(query, k=_req['k'], filter=scope):
            key = doc.metadata['chunk_id']
            sim = 1 - distance
            if key not in best or sim > best[key][1]:
                best[key] = (doc, sim)
    hits = sorted(best.values(), key=lambda pair: pair[1], reverse=True)
    hits = [(doc, sim) for doc, sim in hits if sim >= _req['sim_min']] or hits[:6]
    por_fuente = {}
    diversas = []
    for doc, sim in hits:
        por_fuente.setdefault(doc.metadata['source'], []).append((doc, sim))
    for fuente in sorted(por_fuente, key=lambda f: -max(s for _, s in por_fuente[f])):
        diversas += por_fuente[fuente][:2]
        if len(diversas) >= 6:
            break
    return '\n\n'.join(f'{tag(doc)}\n{doc.page_content[:1200]}' for doc, sim in diversas[:6])

@tool
def contar_documentos(anio: str = ''):
    '''Cuenta los documentos indexados; si se da un año de cuatro digitos, cuenta solo los de ese año.'''
    try:
        manifest = json.loads((Path(os.environ['RAG_INDEX']) / 'manifest.json').read_text())
        names = list(manifest['indexed'])
        return str(sum(1 for name in names if name.startswith(anio)) if anio else len(names))
    except Exception:
        pass
    try:
        got = vectorstore.get(include=['metadatas'])
        names = {m.get('source', '') for m in (got.get('metadatas') or []) if m}
        names = {x for x in names if x}
        if anio:
            names = {x for x in names if x.startswith(anio)}
        return str(len(names))
    except Exception as error:
        return f'No pude contar: {type(error).__name__}'

@tool
def recargar_indice():
    '''Trae el indice mas nuevo de la carpeta compartida de Drive al disco local y lo recarga para buscar. Usala cuando el usuario diga que agrego documentos, cuando contar_documentos no coincida con lo esperado, o antes de una busqueda importante.'''
    import shutil as _shutil
    base = Path(os.environ['RAG_PDFS']).parent
    drive = base / 'index' if str(base).startswith('/content/drive') else None
    if drive is None or not (drive / 'manifest.json').exists():
        return 'No hay indice en Drive para traer.'
    def _n(d):
        m = d / 'manifest.json'
        if not m.exists():
            return -1
        try:
            return len(json.loads(m.read_text()).get('indexed', {}))
        except Exception:
            return -1
    if _n(drive) <= _n(Path(os.environ['RAG_INDEX'])):
        return 'El indice local ya esta al dia.'
    _shutil.rmtree(os.environ['RAG_INDEX'], ignore_errors=True)
    _shutil.copytree(drive, Path(os.environ['RAG_INDEX']))
    global vectorstore
    from langchain_chroma import Chroma as _Chroma
    from langchain_huggingface import HuggingFaceEmbeddings as _HE
    vectorstore = _Chroma(collection_name='rag_core_docs_cosine',
        embedding_function=_HE(model_name=os.environ.get('RAG_EMBED_MODEL', 'sentence-transformers/all-MiniLM-L6-v2')),
        persist_directory=os.environ['RAG_INDEX'], collection_metadata={'hnsw:space': 'cosine'})
    return f'Indice recargado: {vectorstore._collection.count()} chunks disponibles.'
    return str(sum(1 for name in names if name.startswith(anio)) if anio else len(names))


memoria = MemorySaver()

def agent_para(temperatura=0.0, max_tokens=1024, variantes=3):
    llm = ChatOpenAI(base_url=os.environ.get('META_BASE_URL', 'https://api.meta.ai/v1'),
                     api_key=os.environ['META_API_KEY'], model=META_MODEL,
                     temperature=temperatura, max_tokens=int(max_tokens))
    sistema = ('Eres un asistente de investigacion sobre documentos cientificos de denoising en astronomia. '
        f'Tienes buscar_documentos (formula tu hasta {int(variantes)} consultas en ingles tecnico; anio solo si la pregunta menciona un año) y contar_documentos y recargar_indice (trae el indice nuevo de Drive; el agente nunca escribe el indice). '
        'Decide tu cuando usarlas: saludos, gracias, despedidas y charla no necesitan herramientas; para contar usa contar_documentos; '
        'para contenido documental busca primero y responde SOLO con lo que devuelvan las herramientas, citando cada afirmacion con (archivo.pdf, p. N). Resume en maximo 10 vinetas cortas: no pegues fragmentos completos, sintetiza con tus palabras. '
        'Si buscar_documentos no devuelve fragmentos, responde unicamente SIN_EVIDENCIA. '
        'Usa el historial de la conversacion (ahi puede estar el nombre del usuario).')
    return create_agent(llm, tools=[buscar_documentos, contar_documentos, recargar_indice], system_prompt=sistema, checkpointer=memoria)

def ask(question, params=None, thread_id='agent-smoke'):
    params = {**DEFAULTS, **(params or {})}
    _req['k'], _req['sim_min'] = int(params['k']), params['sim_min']
    config = {'configurable': {'thread_id': thread_id}, 'recursion_limit': 12}
    messages = agent_para(params['temperature'], params['max_tokens'], params['variants']).invoke(
        {'messages': [HumanMessage(question)]}, config)['messages']
    turn = messages[max(i for i, m in enumerate(messages) if isinstance(m, HumanMessage)):]
    llamadas = [c['name'] for m in turn for c in (getattr(m, 'tool_calls', None) or [])]
    evidencia = [x for m in turn if isinstance(m, ToolMessage) for x in CITATION.findall(m.content)]
    return {'answer': turn[-1].content.strip(), 'tool_calls': llamadas, 'evidencia': evidencia}

for question in ('Cuantos documentos hay indexados?', 'Que metodos de denoising aparecen en el corpus?'):
    run = ask(question)
    print('PREGUNTA:', question)
    print('Herramientas:', run['tool_calls'] or 'ninguna (directo)')
    print(run['answer'])
    print('Fuentes:', run['evidencia'], '\n')


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

chunks en el indice: 3639
PREGUNTA: Cuantos documentos hay indexados?
Plan: {'intent': 'contar', 'variants': ['Total count of indexed documents available in the search index', 'Number of documents stored in the index for semantic retrieval', 'Indexed collection size and document volume metrics'], 'year': ''}
Hay 47 documentos indexados.
Fuentes: [] 

PREGUNTA: Que metodos aparecen en el corpus?
Plan: {'intent': 'buscar', 'variants': ['Methods described in indexed documents', 'Implementation techniques and procedural steps in indexed documents', 'Methodological frameworks and applied procedures in indexed documents'], 'year': ''}

Fuentes: [] 



## 3. Info del corpus

Totales y tabla por archivo, igual que el panel lateral de la app.

In [4]:
import pandas as pd
rows = []
for name, value in sorted(manifest['indexed'].items()):
    year = name[:4] if re.match(r'(19|20)\d{2}_', name) else 'XXXX'
    if isinstance(value, dict):
        rows.append({'file': name, 'year': year, 'chunks': value.get('chunks', 0), 'pages': value.get('pages', 0)})
    else:
        rows.append({'file': name, 'year': year, 'chunks': value, 'pages': 0})
print(f"archivos: {len(rows)} | fallidos: {len(manifest.get('failed', {}))}")
display(pd.DataFrame(rows))


archivos: 47 | fallidos: 0


,file,year,chunks,pages
0,2021_A_astrovader_astronomical_variational_dee...,2021,123,24
1,2021_A_classification_denoising_cosmic_ray_rad...,2021,29,8
2,2021_A_deepadversaries_examining_robustness_de...,2021,107,20
3,2021_A_learning_denoise_astronomical_images_u_...,2021,60,12
4,2021_B_deep_learning_approach_quasar_continuum...,2021,122,24
5,2021_EA_accelerated_scalable_reproducible_ai_d...,2021,51,12
6,2021_F_self_supervised_denoising_massive_noisy...,2021,42,17
7,2021_H_astronomical_objects_classification_dig...,2021,32,6
8,2021_JE_active_optical_control_machine_learnin...,2021,46,10
9,2021_K_visual_explanation_deep_learning_solar_...,2021,54,11


## 6. Gradio en vivo con streaming

Como la referencia del curso: `ChatInterface` con controles e `debug=True` para ver errores. Enlace público (`share=True`).

## 5. Probar la API directo

Una petición al modelo y una al índice, sin interfaz. Si algo falla aquí, es clave/modelo/índice — no Gradio.

In [ ]:
# Prueba directa a la API (antes de la interfaz): si esto falla, es la clave/modelo/endpoint, no Gradio
proba = ChatOpenAI(base_url=os.environ.get('META_BASE_URL', 'https://api.meta.ai/v1'),
                   api_key=os.environ['META_API_KEY'],
                   model=os.environ.get('META_MODEL', 'muse-spark-1.3-contributor'), temperature=0)
print('API dice:', proba.invoke('Saluda en una sola frase.').content)

# Prueba de recuperacion: la herramienta responde sin llamar al modelo
print(buscar_documentos.invoke({'consultas': ['astronomical image denoising'], 'anio': ''})[:600])


In [6]:
import gradio as gr
from langchain_core.messages import AIMessageChunk

_hilos = {}

def responder(mensaje, historial, temperatura, variantes, k, sim_min, max_tokens, request: gr.Request):
    sesion = request.session_hash
    if not historial:
        _hilos[sesion] = _hilos.get(sesion, 0) + 1
    thread = f"{sesion}-{_hilos.get(sesion, 0)}"
    _req['k'], _req['sim_min'] = int(k), sim_min
    agente = agent_para(temperatura, int(max_tokens), int(variantes))
    config = {'configurable': {'thread_id': thread}, 'recursion_limit': 12}
    print('preg:', mensaje)
    yield 'Buscando en el corpus...'
    try:
        respuesta = ''
        for trozo, info in agente.stream({'messages': [HumanMessage(mensaje)]}, config, stream_mode='messages'):
            if not isinstance(trozo, AIMessageChunk) or getattr(trozo, 'tool_calls', None):
                continue
            txt = getattr(trozo, 'text', None)
            if not txt and isinstance(getattr(trozo, 'content', ''), str):
                txt = trozo.content
            if not txt:
                continue
            respuesta += txt
            yield respuesta
        print('respuesta chars:', len(respuesta))
        print('---- respuesta ----')
        print(respuesta)
        print('-------------------')
        if not respuesta.strip():
            out = agente.invoke({'messages': [HumanMessage(mensaje)]}, config)
            respuesta = out['messages'][-1].content.strip() or 'SIN_EVIDENCIA'
            print('fallback sync chars:', len(respuesta))
            print('---- respuesta (fallback) ----')
            print(respuesta)
            print('-------------------')
            yield respuesta
    except Exception as error:
        import traceback as _tb
        _tb.print_exc()
        yield f'Error en el agente: {type(error).__name__}: {error}'
controles = [

    gr.Slider(0, 1.5, value=DEFAULTS['temperature'], step=0.05, label='Temperatura'),
    gr.Slider(1, 5, value=DEFAULTS['variants'], step=1, label='Consultas por pregunta'),
    gr.Slider(1, 8, value=DEFAULTS['k'], step=1, label='Fragmentos por consulta (k)'),
    gr.Slider(0, 0.9, value=DEFAULTS['sim_min'], step=0.05, label='Umbral de similitud'),
    gr.Slider(128, 4096, value=DEFAULTS['max_tokens'], step=128, label='Máximo de tokens'),
]
ejemplos = [
    ['¿Cuántos documentos hay indexados?', 0.0, 3, 4, 0.35, 2048],
    ['¿Qué métodos de denoising aparecen en el corpus?', 0.0, 3, 4, 0.35, 2048],
    ['¿Qué métricas usan para evaluar el denoising?', 0.0, 3, 4, 0.35, 2048],
]

demo = gr.ChatInterface(
    fn=responder,
    title='Agente RAG sobre documentos',
    description='Pregunta sobre los PDF del corpus. Citas con (archivo.pdf, p. N); SIN_EVIDENCIA si no hay soporte.',
    additional_inputs=controles,
    additional_inputs_accordion='Hiperparámetros',
    examples=ejemplos,
)
demo.launch(theme=gr.themes.Ocean(), share=True, debug=True)


/tmp/ipykernel_3709/54925900.py:36: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(title='Agente RAG sobre documentos', theme=THEME, css=CSS) as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://a79ae5766dfb8bd9ef.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
